<center><p float="center">
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<h1><center> Real-Time Retail Feedback Intelligence
 </center></h1>


### **Business Context**
Why is this problem important to solve?

ChicStyle is a growing fashion retail platform that sees massive spikes in customer activity during festive seasons and holiday sales. As shoppers buy clothing and accessories for celebrations, the volume of incoming reviews rises sharply — they arrive every hour, ranging from positive praise to urgent complaints about fit, delivery delays, product defects, or sizing issues. During these high-pressure periods, even a slight delay in reading or responding to feedback can have serious consequences. If the retail team fails to act quickly, customers feel ignored at an emotionally significant time, which leads to frustration, spoiled shopping experiences, and reduced trust in the brand. The cost is twofold: immediate lost sales, and longer-term damage to repeat purchases and loyalty. To protect both revenue and reputation at peak periods, ChicStyle needs a system that can process thousands of reviews instantly, accurately, and with business context.

Customer reviews are direct, unsolicited signals about what is working and what is failing across the product and service experience. Analyzing them well lets a retailer:

- **Catch problems early** — defects, sizing inconsistencies, and delivery failures surface in reviews before they show up in returns and churn.
- **Prioritize action** — separating urgent complaints from minor suggestions lets limited support and operations capacity focus where it matters most.
- **Improve products faster** — recurring themes (e.g., a dress that runs small) feed directly back to merchandising and quality teams.
- **Protect brand reputation and loyalty** — responding promptly, especially to negative feedback, signals that the brand listens and converts a bad experience into a recoverable one.
- **Turn unstructured text into business intelligence** — at scale, reviews become a measurable, trackable input to decisions rather than anecdotes.

The challenge is volume and nuance. Traditional rule-based NLP struggles with mixed feedback — for example, *"The fit is great but the color was not as per the product image"* contains both a positive and a negative opinion, and older systems tend to collapse it into a single sentiment, losing critical detail. This is precisely the gap a Generative AI approach is meant to close.


### **Objective**
What is the intended goal?

Build a **Generative AI feedback system** that uses prompt engineering — **Zero-Shot, Few-Shot, and Chain-of-Thought (CoT)** prompting — to transform large volumes of raw customer reviews into structured, actionable intelligence. Specifically, the system should:

- **Analyze and categorize sentiment in real time** (positive, negative, neutral), including mixed feedback.
- **Detect which product or service** each piece of feedback refers to (e.g., fit, quality, delivery, price), even though there is no explicit service column — it is inferred from the review text.
- **Summarize insights by product category and urgency level**, where category comes from existing department data and urgency (high / medium / low) is inferred from the language of the review.
- **Automatically send short, personalized messages** to customers based on sentiment — thanking them for positive feedback, acknowledging neutral comments, and apologizing for negative ones while letting them know a team member will follow up.
- **Generate short, actionable reports** for retail teams.

Manual review reading does not scale to thousands of reviews per hour during peak sales, and it is slow, inconsistent, and expensive. Automating it lets ChicStyle:

- **Respond in real time** rather than hours or days later, so customers feel heard during emotionally important purchases.
- **Act on issues quickly** — high-urgency complaints are flagged and routed instead of being buried in a backlog.
- **Improve product quality faster** by surfacing recurring themes automatically.
- **Apply consistent, context-aware judgment** at scale, capturing nuance that rule-based models miss.
- **Free up human teams** to focus on resolution and relationships instead of triage.

In short, the goal is to turn massive unstructured feedback into meaningful, real-time business intelligence that protects revenue, reputation, and loyalty during the moments that matter most.

### **Dataset Used for the Notebook**
Describe dataset used for this project.

#### **Description**

The project uses the **"Women's E-Commerce Clothing Reviews"** dataset: **23,486 reviews with 10 columns**. 
Each row is a single customer review of a clothing item, combining structured metadata (rating, recommendation, department) with unstructured review text.

| Column                  | Description                                                  | Type             |
| ----------------------- | ------------------------------------------------------------ | ---------------- |
| Clothing.ID             | Unique ID for each piece of clothing                         | Integer          |
| Age                     | Age of the reviewer                                          | Positive integer |
| Title                   | Title of the review                                          | String           |
| Review.Text             | Main body of the review                                      | String           |
| Rating                  | Product score, 1 (worst) to 5 (best)                         | Ordinal integer  |
| Recommended.IND         | Whether the customer recommends the product (1 = yes, 0 = no) | Binary           |
| Positive.Feedback.Count | Number of other customers who found the review helpful       | Positive integer |
| Division.Name           | High-level division of the product                           | Categorical      |
| Department.Name         | Specific department of the product                           | Categorical      |
| Class.Name              | Product class (finer than department)                        | Categorical      |

Reviewers skew middle-aged: ages range from **18 to 99** with a **mean of ~43** and a median of 41. Reviews average about **60 words** each.

> **Note on scope:** For the GenAI prompt-engineering and evaluation work, only a **sample of 50 reviews** is processed (5–10 during initial prompt testing, scaling to 50 for final evaluation) to stay within the API budget. The full 23,486-row dataset is only used for the exploratory analysis.

#### **Key Patterns Summary**
*Ratings — strongly skewed positive:*

| Rating | Count  | Share |
| ------ | ------ | ----- |
| 5      | 13,131 | 55.9% |
| 4      | 5,077  | 21.6% |
| 3      | 2,871  | 12.2% |
| 2      | 1,565  | 6.7%  |
| 1      | 842    | 3.6%  |

About **77.5% of reviews are 4–5 stars** and **82.2% of customers recommend** the product. This class imbalance matters: negative reviews are the minority but carry the highest business urgency, so the system must not let the volume of praise drown out the smaller stream of complaints. Rating aligns tightly with recommendation — the average rating is **4.6 among recommenders vs. 2.3 among non-recommenders** — confirming the two signals are consistent.

*Departments — concentrated in a few categories:*

| Department | Count  |
| ---------- | ------ |
| Tops       | 10,468 |
| Dresses    | 6,319  |
| Bottoms    | 3,799  |
| Intimate   | 1,735  |
| Jackets    | 1,032  |
| Trend      | 119    |

**Tops and Dresses together account for ~71%** of all reviews, so feedback volume — and therefore the operational load and the biggest opportunity for product improvement — is concentrated there. At the division level, **General (13,850)** and **General Petite (8,120)** dominate, with Intimates smallest. The finest level (Class.Name) is led by Dresses, Knits, and Blouses.

*Fit and sizing dominate both ends:*

- **Positive reviews (4–5★):** *love, great, perfect, fit, size, fabric, color, dress, top, comfortable* — praise centers on fit, fabric, and overall delight.
- **Negative reviews (1–2★):** *too, small, fabric, fit, size, ordered, look, back* — complaints cluster around **fit/sizing** ("too small"), **fabric quality**, and **returns** ("ordered… back").

The recurring takeaway is that **fit and sizing are the single most important theme across the entire experience** — they drive both the strongest praise and the strongest complaints. Fabric/quality and color-vs-image expectations are the next most common pain points.

#### **Data treatments / preprocessing required**
- **Missing values:** `Review.Text` is missing in **845 rows** and `Title` in **3,810 rows**; `Division.Name`, `Department.Name`, and `Class.Name` are each missing in **14 rows**. Rows with no review text cannot be analyzed by the language model and should be dropped (or excluded from the sampled set); the 14 rows missing category fields can be dropped or imputed as "Unknown."
- **CSV parsing:** the file is **semicolon-delimited** with an unnamed leading index column — it must be read with the correct separator and index handling, and review text contains embedded quotes/commas that must be parsed cleanly.
- **Text cleaning for analysis:** for EDA and word clouds, lowercase the text, strip punctuation, and remove stopwords; keep the raw text intact for the LLM so it retains nuance.
- **Sampling:** draw a representative ~50-review sample for prompt experiments, ideally spanning the rating range and major departments so evaluation isn't dominated by 5-star Tops reviews.
- **Type checks:** confirm `Rating` (1–5) and `Recommended.IND` (0/1) fall in valid ranges; treat `Rating`, `Department.Name`, and `Division.Name` as categ/ordinal as appropriate.
- **No new labels needed for category/urgency:** the project intentionally does **not** add sentiment, "Category," or "urgency" columns to the data — these are generated by the LLM from the review text at analysis time, with product category drawn from the existing `Department.Name`.

### **Installing and Importing Necessary Libraries**
First, let's set up the environment by installing the required Python libraries.

In [2]:
# Install the required libraries for the project

In [3]:
# Import the required libraries for the project

# --- Core Utilities ---
import time
import json
import re
import numpy as np
from typing import Dict, List, Optional, Any, Tuple

# --- Data Handling ---
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

# --- Machine Learning Metrics ---
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# --- Progress Bar ---
from tqdm import tqdm

# --- OpenAI Client ---
import openai

# Set global plot style
sns.set_style("whitegrid")


### **Data Loading**
### Loading and Understanding the Data


In [ ]:
### Loading and Understanding the Data
# Load the dataset from 'data.csv'. 
# Note: the separator is a semicolon (';') and the first column should be used as the index.

df = pd.read_csv(_________, sep=_________, index_col=_________)
display(df)

,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...
23482,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses
23483,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits
23484,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses
23485,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses


### **Data Overview**

Dataset Head:


,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


### **Sanity checks**


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 23486 entries, 1 to 23486
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Clothing.ID              23486 non-null  int64 
 1   Age                      23486 non-null  int64 
 2   Title                    19676 non-null  object
 3   Review.Text              22641 non-null  object
 4   Rating                   23486 non-null  int64 
 5   Recommended.IND          23486 non-null  int64 
 6   Positive.Feedback.Count  23486 non-null  int64 
 7   Division.Name            23472 non-null  object
 8   Department.Name          23472 non-null  object
 9   Class.Name               23472 non-null  object
dtypes: int64(5), object(5)
memory usage: 2.0+ MB


### **Data Cleaning and Preprocessing**

**Think about it:** The Review Text column is the most critical feature for our Generative AI model. What should be done with rows where this text is missing?

### **Exploratory Data Analysis**

EDA is an important part of any project involving data. It is important to investigate and understand the data better before building a model with it. A few questions have been mentioned below which will help you approach the analysis in the right manner and generate insights from the data. A thorough analysis of the data, in addition to the questions mentioned below, should be done.

**Questions:**

1.  What is the summary statistics of the numerical data? What can you infer about the distribution of Age, Rating, and Positive Feedback Count?
    
2.  How many unique values are there in the categorical columns like Division Name, Department Name, and Class Name?
    
3.  What is the overall distribution of product Rating? Is the dataset skewed towards positive or negative reviews?
    
4.  Which Department Name receives the highest average rating, and which receives the lowest? What might this indicate?
    
5.  What are the most common words found in highly-rated reviews (4-5 stars) versus poorly-rated reviews (1-2 stars)? (Hint: Use Word Clouds). What initial hypotheses can you form about the key drivers of customer satisfaction and dissatisfaction?

Also write your observations for each questions.

## **Building the Generative AI Pipeline**

We will now build a system to analyze the reviews. This involves setting up the AI client, designing prompts, generating structured data, and evaluating the results.

#### **Setup AI Client and Data Sample**

**Questions:**

1.  How do you initialize the OpenAI client with your API key and the correct base URL?
    

#### **Note:**

For this project, we will analyze and categorize a sample of **50 customer reviews**. This number is chosen intentionally. Since the API has a **budget limit of $20**, running prompts on very large datasets can quickly exhaust your quota—especially because this exercise may involve **multiple iterations, prompt refinements, and repeated evaluations**.

To avoid unnecessary cost and ensure efficient experimentation, we recommend the following approach:

*   **Use very small samples (5–10 reviews)** during the **initial testing phase** to validate your prompt structure and logic.
    
*   **Scale up to 50 reviews** for the **final evaluation phase**, ensuring you get enough data to compare prompting techniques without draining your budget.
    
*   This strategy helps maintain cost control while still providing meaningful insights across Zero-Shot, Few-Shot, and Chain-of-Thought techniques.
    

If your API quota gets exhausted, you may temporarily switch to another free AI assistant API. However, note that external tools may also have **rate limits** or **token caps**, so you will need to build retry logic and manage throttling within your code.

#### **Prompt Engineering and Evaluation**

We will test three different prompting techniques. For each, we will create a basic version (V1) and an enhanced version (V2).

**Think about it:** Why is it important to have a consistent and robust evaluation framework? How can we use an "LLM-as-Judge" to score the quality of our generated outputs objectively?

#### **Technique 1: Zero-Shot Prompting**

**Questions:**

1.  How would you design a basic Zero-Shot prompt that asks the model for Category, Sentiment, Summary, Personalized Message, and Retail Insight?
    
2.  How can you enhance this prompt with more business context (e.g., a company name, the importance of accuracy) to create a V2 prompt?
    
3.  How will you loop through the data sample to generate and store the structured output for both prompt versions?
    
4.  How will you apply the LLM-as-Judge to generate a evaluation score between 0 to 1 (decimal allowed) for the outputs and calculate the average score of V1 and V2 prompt?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Zero-Shot Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

#### **Technique 2: Few-Shot Prompting**

**Questions:**

1.  How do you structure a Few-Shot prompt? What kind of examples (e.g., one positive, one negative) would be most effective?
    
2.  For the V2 prompt, how can you add a set of "rules" to guide the model's output for each field, reducing ambiguity?
    
3.  After generating and scoring the outputs, how does the performance of Few-Shot prompting compare to previous version?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your ** Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

#### **Technique 3: Chain-of-Thought (CoT) Prompting**

**Questions:**

1.  How do you instruct the model to "think step-by-step" internally but only show the final, structured answer?
    
2.  How can you combine the CoT instruction with more detailed reasoning from the COT V1 prompt to create a powerful CoT V2 prompt?
    
3.  Does encouraging the model to reason first lead to a measurable improvement in the quality of the generated insights?

**How the process works:**

1.  First, you create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.
    
2.  Then, you run your **Prompt Version 1** on a sample of 50 reviews to generate predictions.
    
3.  You use the judge function to **score each prediction** and compute the **average score for Version 1**.
    
4.  Next, you repeat the same workflow with your **Version 2 prompt**, generate predictions, evaluate them, and calculate the **average score for Version 2**.

## **Applying GenAI for Product Recommendation:**

Now, let's use the model for a different task: predicting the Recommended IND flag.

**Questions:**

1.  How do you design a prompt that strictly asks for a binary output (1 or 0) and a brief reason?
    
2.  What kind of function is needed to reliably parse the model's text response to extract the 1/0 flag and the Reason?
    
3.  How do you evaluate the model's performance as a classifier using standard metrics like accuracy, confusion matrix, and classification report?

**How the Process Works**


**1\. Prepare Data**

Copy the dataset, store the original recommendation labels, and remove them from the model input to avoid leakage.

**2\. Generate Predictions**

Use a strict two-line prompt to make the LLM output a binary recommendation (1/0) and a short reason based only on the review text.

**3\. Parse Outputs**

Extract the flag and reason from the raw LLM response using regex-based parsing that handles formatting issues.

**4\. Build Prediction Table**

Run the prompt for each review, parse the result, and store the predictions in a new DataFrame.

 **5\. Evaluate Performance**

Compare LLM predictions with true labels using accuracy, confusion matrix, and classification report.

 **6\. Explain Mismatches**

For incorrect predictions, generate a short explanation describing why the model’s decision may have differed from the human label.

**Visualization of Sentiments Distribution**

 After generating results from all prompting techniques, it's crucial to visualize their outputs to better understand their behavior and performance. This helps us see if one technique tends to be more cautious (e.g., assigning more 'Neutral' sentiments) or if they generally agree on the sentiment of the reviews.
    
 **Questions:**
    
* How does the distribution of predicted Sentiment (Positive, Negative, Neutral) compare across the V2 versions of Zero-Shot, Few-Shot, and Chain-of-Thought? (Hint: Create a separate bar chart for each technique's V2 sentiment column).
    
* Are there noticeable differences in the counts? For example, does one technique identify more "Neutral" reviews than the others? What might this imply about its ability to handle nuance?
    


##  **Comparison of Prompting Techniques:**
    
   *   How do the three techniques (Zero-Shot, Few-Shot, CoT) compare in terms of their responses. Use LLM to give verdict?
        
  *   Which technique was the most reliable and consistent? Why do you think it performed the best?
        
   *   What model and prompt design would you propose for a production environment?
        


### **Observations and Insights**

 **Refined Insights:**
    
   *   What are the most meaningful and recurring insights from the customer reviews, as identified by your best-performing model?

# Generating Actionable Product Improvement Suggestions


 *   Based on the aggregated insights from your best model, what are 3 short-term (3-6 months) and 3 long-term (6-12 months) actionable business recommendations for the retail company?
        
 *   How does this automated GenAI pipeline solve the initial business problem and create value?

### **Observations and Insights**

## **Conclusion**